# Session 2: one pixel grid, one time base
**A. Real-world problem.** Camera A is 4:3 and sees the sample head-on; camera B is 16:9, looks
from the side, and runs at a different frame rate. Before any correlation, both must be
turned into the same kind of data: the same pixel grid and the same clock.

**Plan.** (1) solve a homography from 4 point pairs, (2) warp an image with it,
(3) cost the aspect-ratio choices, (4) measure resampling blur, (5) build Akima and Makima
interpolation and compare them with linear and cubic spline, (6) find a timing offset.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: a homography from four point pairs
A homography maps a flat surface seen by one camera onto the same surface seen by another:

    x' = (h11 x + h12 y + h13) / (h31 x + h32 y + 1)
    y' = (h21 x + h22 y + h23) / (h31 x + h32 y + 1)

Eight unknowns (h33 is fixed to 1). Each point pair gives two equations, so four pairs give
an 8 x 8 linear system. Multiply out the denominator and the equations become linear in h.

In [ ]:
def homography_from_points(src, dst):
    """3x3 matrix H (h33 = 1) that maps each src point (x, y) to its dst point (u, v)."""
    src = np.asarray(src, dtype=float)
    dst = np.asarray(dst, dtype=float)
    if src.shape != (4, 2) or dst.shape != (4, 2):
        raise ValueError("need exactly four point pairs (4 x 2 arrays)")
    A = np.zeros((8, 8))
    b = np.zeros(8)
    for i in range(4):
        x, y = src[i]
        u, v = dst[i]
        A[2 * i] = [x, y, 1, 0, 0, 0, -x * u, -y * u]
        A[2 * i + 1] = [0, 0, 0, x, y, 1, -x * v, -y * v]
        b[2 * i], b[2 * i + 1] = u, v
    try:
        if np.linalg.cond(A) > 1e12:
            raise np.linalg.LinAlgError
        h = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        raise ValueError("the four points are degenerate (three on a line?)") from None
    return np.append(h, 1.0).reshape(3, 3)


def apply_h(H, pts):
    """Apply a homography to an (N, 2) array of points."""
    pts = np.atleast_2d(np.asarray(pts, dtype=float))
    q = np.hstack([pts, np.ones((len(pts), 1))]) @ np.asarray(H).T
    return q[:, :2] / q[:, 2:3]

## Step 2: warp an image by inverse mapping
To build the output image we do NOT push source pixels forward (that leaves holes). For each
OUTPUT pixel we ask: where in the source did you come from? That is H inverse. The source
position is usually between pixels, so we read it with bilinear interpolation: a weighted
average of the four neighbouring pixels. Pixel centres sit at integer coordinates (x = column,
y = row). Output pixels that map outside the source are NaN ("no data").

In [ ]:
def bilinear(img, x, y):
    """Sample img at float positions (x = column, y = row). NaN outside the image."""
    h, w = img.shape
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    inside = (x >= 0) & (x <= w - 1) & (y >= 0) & (y <= h - 1)
    xs = np.where(inside, x, 0.0)
    ys = np.where(inside, y, 0.0)
    x0 = np.clip(np.floor(xs).astype(int), 0, w - 2)
    y0 = np.clip(np.floor(ys).astype(int), 0, h - 2)
    fx, fy = xs - x0, ys - y0
    val = ((1 - fx) * (1 - fy) * img[y0, x0] + fx * (1 - fy) * img[y0, x0 + 1]
           + (1 - fx) * fy * img[y0 + 1, x0] + fx * fy * img[y0 + 1, x0 + 1])
    return np.where(inside, val, np.nan)


def warp_image(img, H, shape):
    """Image of shape (rows, cols) on the output grid; H maps source (x, y) to output (x, y)."""
    rows, cols = shape
    gy, gx = np.mgrid[0:rows, 0:cols].astype(float)
    pts = np.stack([gx.ravel(), gy.ravel()], axis=1)
    src = apply_h(np.linalg.inv(H), pts)
    return bilinear(img, src[:, 0], src[:, 1]).reshape(shape)


def resize_matrix(src_shape, dst_shape):
    """H that stretches the whole source image onto the whole destination image (corners to corners)."""
    sy, sx = dst_shape[0] / src_shape[0], dst_shape[1] / src_shape[1]
    return np.array([[sx, 0, 0.5 * sx - 0.5], [0, sy, 0.5 * sy - 0.5], [0, 0, 1.0]])

## Step 3: a speckle pattern we can sample anywhere
DIC needs a random dot pattern. We build one from 48 random cosine waves, so it can be
evaluated at any point (no pixel grid needed) and has a typical dot size of a few pixels.

In [ ]:
def speckle_waves(seed=7, count=48, kmin=0.35, kmax=1.2):
    rng = np.random.default_rng(seed)
    k = rng.uniform(kmin, kmax, count)
    th = rng.uniform(0, 2 * np.pi, count)
    return np.stack([k * np.cos(th), k * np.sin(th), rng.uniform(0, 2 * np.pi, count)], axis=1)


def speckle_at(waves, x, y):
    """Pattern value (mean 0, standard deviation about 1) at float positions."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    out = np.zeros(x.shape)
    for kx, ky, ph in waves:
        out += np.cos(kx * x + ky * y + ph)
    return out * np.sqrt(2.0 / len(waves))


def speckle_image(rows, cols, seed=7):
    gy, gx = np.mgrid[0:rows, 0:cols].astype(float)
    return speckle_at(speckle_waves(seed), gx, gy)

## Step 4: the cost of fitting one aspect ratio into another
Source w x h pixels, target grid W x H. Three ways to fit:
letterbox (shrink until everything fits, leave empty bars), crop (shrink until the grid is
full, throw away the overflow) and stretch (different scale in x and y).

In [ ]:
def aspect_costs(src_w, src_h, grid_w, grid_h):
    """Cost of each fit as fractions: empty target pixels, lost source pixels, and x:y stretch ratio."""
    r_src, r_grid = src_w / src_h, grid_w / grid_h
    lo, hi = min(r_src, r_grid), max(r_src, r_grid)
    return {"letterbox_empty": 1.0 - lo / hi, "crop_lost": 1.0 - lo / hi, "stretch_ratio": hi / lo}


def resample_roundtrip_error(scale, size=160, seed=7):
    """Resize a speckle image by `scale` and back (bilinear both ways).
    Returns RMS difference divided by the pattern's standard deviation (0 = perfect)."""
    img = speckle_image(size, size, seed)
    small_shape = (int(round(size * scale)), int(round(size * scale)))
    s = small_shape[0] / size
    down = warp_image(img, resize_matrix(img.shape, small_shape), small_shape)
    back = warp_image(down, resize_matrix(small_shape, img.shape), img.shape)
    m = 8
    a, b = img[m:-m, m:-m], back[m:-m, m:-m]
    ok = np.isfinite(b)
    return float(np.sqrt(np.mean((a[ok] - b[ok]) ** 2)) / np.std(img)), s


def shift_std_ratio(frac, size=400, seed=1):
    """Shift white noise by `frac` pixel with bilinear sampling; ratio of std after/before."""
    img = np.random.default_rng(seed).normal(size=(size, size))
    shifted = warp_image(img, np.array([[1.0, 0, frac], [0, 1.0, 0], [0, 0, 1.0]]), img.shape)
    ok = np.isfinite(shifted)
    return float(np.std(shifted[ok]) / np.std(img[ok]))

## Step 5: the time axis. Linear, cubic spline and Makima
Camera A gives a displacement sample every 1/30 s, camera B every 1/24 s. Both go onto a
common 120 fps timeline (120 = 4 x 30 = 5 x 24, so every real sample lands on a timeline tick).
The values in between must be interpolated.

**Makima** (modified Akima) builds a cubic Hermite curve: through every sample, with a slope
d_i chosen at each sample. Let m_i be the slope of the straight segment from sample i to i+1.
The slope at sample i is a weighted average of its two neighbouring segment slopes:

    d_i = (w1 m_{i-1} + w2 m_i) / (w1 + w2)
    w1 = |m_{i+1} - m_i| + |m_{i+1} + m_i| / 2
    w2 = |m_{i-1} - m_{i-2}| + |m_{i-1} + m_{i-2}| / 2

The side whose neighbouring slopes differ most gets the larger weight on the OTHER segment,
so a bend on one side does not drag the slope. The extra |m + m'| / 2 term is the "modified"
part: the original Akima weight is only |m' - m|, which is exactly 0 whenever two neighbouring
slopes are equal, and then the formula cannot decide (0/0). Makima keeps the weights above
zero unless the data are completely flat, and picks the smaller slope at a corner between a
flat run and a rising run, so it does not overshoot. For equally spaced samples lying on a
straight line all m are equal, so d_i = m: lines are reproduced exactly.

In [ ]:
def _extend_slopes(m):
    """Segment slopes m_0..m_{n-2} extended by two ghost slopes at each end (linear extrapolation)."""
    n = len(m) + 1
    e = np.empty(n + 3)
    e[2:n + 1] = m
    e[1] = 2 * m[0] - m[1]
    e[0] = 2 * e[1] - m[0]
    e[n + 1] = 2 * m[-1] - m[-2]
    e[n + 2] = 2 * e[n + 1] - m[-1]
    return e


def akima_slopes(x, y, modified=True):
    """Slopes at every sample for Akima (modified=False) or Makima (modified=True)."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    m = np.diff(y) / np.diff(x)
    if n == 2:
        return np.array([m[0], m[0]])
    e = _extend_slopes(m)
    # for sample i the four slopes m_{i-2}, m_{i-1}, m_i, m_{i+1} sit at e[i], e[i+1], e[i+2], e[i+3]
    m2, m1, p0, p1 = e[0:n], e[1:n + 1], e[2:n + 2], e[3:n + 3]
    w1 = np.abs(p1 - p0)
    w2 = np.abs(m1 - m2)
    if modified:
        w1 = w1 + 0.5 * np.abs(p1 + p0)
        w2 = w2 + 0.5 * np.abs(m1 + m2)
    tot = w1 + w2
    safe = np.where(tot > 0, tot, 1.0)
    return np.where(tot > 0, (w1 * m1 + w2 * p0) / safe, 0.5 * (m1 + p0))


def hermite_eval(x, y, d, xq, deriv=0, extrapolate=False):
    """Piecewise cubic Hermite curve through (x, y) with slopes d. deriv = 0, 1 or 2.
    Outside [x[0], x[-1]] the result is NaN unless extrapolate=True (then the end cubic continues)."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    xq = np.asarray(xq, dtype=float)
    i = np.clip(np.searchsorted(x, xq, side="right") - 1, 0, len(x) - 2)
    h = x[i + 1] - x[i]
    t = (xq - x[i]) / h
    y0, y1, d0, d1 = y[i], y[i + 1], d[i], d[i + 1]
    if deriv == 0:
        out = ((2 * t**3 - 3 * t**2 + 1) * y0 + (t**3 - 2 * t**2 + t) * h * d0
               + (-2 * t**3 + 3 * t**2) * y1 + (t**3 - t**2) * h * d1)
    elif deriv == 1:
        out = ((6 * t**2 - 6 * t) * y0 / h + (3 * t**2 - 4 * t + 1) * d0
               + (-6 * t**2 + 6 * t) * y1 / h + (3 * t**2 - 2 * t) * d1)
    else:
        out = ((12 * t - 6) * y0 / h**2 + (6 * t - 4) * d0 / h
               + (-12 * t + 6) * y1 / h**2 + (6 * t - 2) * d1 / h)
    if not extrapolate:
        out = np.where((xq < x[0]) | (xq > x[-1]), np.nan, out)
    return out


def makima(x, y, xq, deriv=0, extrapolate=False):
    return hermite_eval(x, y, akima_slopes(x, y, True), xq, deriv, extrapolate)


def akima(x, y, xq, deriv=0, extrapolate=False):
    return hermite_eval(x, y, akima_slopes(x, y, False), xq, deriv, extrapolate)


def spline_second_derivs(x, y):
    """Second derivatives of the not-a-knot cubic spline (the Matlab default 'spline')."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    h = np.diff(x)
    dl = np.diff(y) / h
    A = np.zeros((n, n))
    r = np.zeros(n)
    A[0, 0:3] = [h[1], -(h[0] + h[1]), h[0]]
    A[n - 1, n - 3:n] = [h[n - 2], -(h[n - 3] + h[n - 2]), h[n - 3]]
    for i in range(1, n - 1):
        A[i, i - 1:i + 2] = [h[i - 1], 2 * (h[i - 1] + h[i]), h[i]]
        r[i] = 6 * (dl[i] - dl[i - 1])
    return np.linalg.solve(A, r)


def spline(x, y, xq, extrapolate=False):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    xq = np.asarray(xq, dtype=float)
    M = spline_second_derivs(x, y)
    i = np.clip(np.searchsorted(x, xq, side="right") - 1, 0, len(x) - 2)
    h = x[i + 1] - x[i]
    a, b = x[i + 1] - xq, xq - x[i]
    out = (M[i] * a**3 / (6 * h) + M[i + 1] * b**3 / (6 * h)
           + (y[i] / h - M[i] * h / 6) * a + (y[i + 1] / h - M[i + 1] * h / 6) * b)
    if not extrapolate:
        out = np.where((xq < x[0]) | (xq > x[-1]), np.nan, out)
    return out


def linear(x, y, xq, extrapolate=False):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    xq = np.asarray(xq, dtype=float)
    i = np.clip(np.searchsorted(x, xq, side="right") - 1, 0, len(x) - 2)
    t = (xq - x[i]) / (x[i + 1] - x[i])
    out = (1 - t) * y[i] + t * y[i + 1]
    if not extrapolate:
        out = np.where((xq < x[0]) | (xq > x[-1]), np.nan, out)
    return out


METHODS = {"linear": linear, "spline": spline, "akima": akima, "makima": makima}


def interpolate(method, t, y, tq, extrapolate=False):
    """Resample the series (t, y) at times tq with 'linear', 'spline', 'akima' or 'makima'."""
    return METHODS[method](t, y, tq, extrapolate=extrapolate)

## Step 6: two motions to test with, and the error against the truth
Smooth motion: two sine waves (pixels). Step-like motion: a 4 px jump over about 40 ms,
the kind of thing a clamp release or a crack opening does.

In [ ]:
def smooth_motion(t):
    t = np.asarray(t, dtype=float)
    return 2.0 * np.sin(2 * np.pi * 1.5 * t) + 0.8 * np.sin(2 * np.pi * 3.1 * t + 0.6)


def step_motion(t, t0=0.52, width=0.01, amp=4.0):
    t = np.asarray(t, dtype=float)
    return amp * 0.5 * (1 + np.tanh((t - t0) / width))


def timeline(fps=120, t_end=2.0):
    return np.arange(int(round(t_end * fps)) + 1) / fps


def camera_samples(signal, fps, t_end=2.0):
    t = np.arange(int(round(t_end * fps)) + 1) / fps
    return t, signal(t)


def rms_vs_truth(signal, method, fps, window=(0.0, 2.0), t_end=2.0, grid_fps=120):
    """RMS error (px) of a camera running at `fps`, resampled onto the common timeline,
    measured only for times inside `window` (seconds)."""
    t, y = camera_samples(signal, fps, t_end)
    tl = timeline(grid_fps, t_end)
    tl = tl[(tl >= window[0] - 1e-9) & (tl <= window[1] + 1e-9)]
    err = interpolate(method, t, y, tl) - signal(tl)
    return float(np.sqrt(np.nanmean(err**2)))


STEP_WINDOW = (0.3, 0.8)     # the part of the timeline around the jump


def max_overshoot(method, signal, fps, window=STEP_WINDOW, lo=0.0, hi=4.0, n=5001):
    """How far (px) the interpolated curve goes outside the true range [lo, hi] inside `window`."""
    t, y = camera_samples(signal, fps)
    tq = np.linspace(window[0], window[1], n)
    v = interpolate(method, t, y, tq)
    return float(max(np.nanmax(v) - hi, lo - np.nanmin(v), 0.0))


def corner_dip(method):
    """Lowest value of the curve through a flat run that turns into a ramp: y = 0 0 0 1 2 3.
    The true shape never goes below 0."""
    x = np.arange(6.0)
    y = np.array([0.0, 0.0, 0.0, 1.0, 2.0, 3.0])
    return float(np.min(interpolate(method, x, y, np.linspace(0, 5, 2001))))

## Step 7: the timing offset between the cameras
The two cameras never start on the same tick. If camera B's clock is late by tau, B's sample
labelled t really recorded the motion at t + tau. After putting both on the 120 fps timeline
we slide one against the other (cross-correlation) and read the lag of the best match; a
parabola through the three points around the peak gives a fraction of a tick.

In [ ]:
def wander_motion(t):
    t = np.asarray(t, dtype=float)
    return 3.0 * np.sin(2 * np.pi * 0.7 * t) + 2.0 * np.exp(-((t - 1.0) / 0.25) ** 2) + 1.2 * np.sin(2 * np.pi * 2.2 * t + 1.0)


def estimate_offset(tA, yA, tB, yB, method="makima", fs=120, max_lag=0.1):
    """Offset tau (s) such that B(t) = A(t + tau), from the cross-correlation of the two resampled series."""
    lo, hi = max(tA[0], tB[0]), min(tA[-1], tB[-1])
    tl = np.arange(int(np.ceil(lo * fs)), int(np.floor(hi * fs)) + 1) / fs
    a = interpolate(method, tA, yA, tl)
    b = interpolate(method, tB, yB, tl)
    K = int(round(max_lag * fs))
    N = len(tl)
    bs = b[K:N - K] - np.mean(b[K:N - K])
    c = []
    for k in range(-K, K + 1):
        as_ = a[K + k:N - K + k]
        as_ = as_ - np.mean(as_)
        c.append(float(np.dot(as_, bs) / np.sqrt(np.dot(as_, as_) * np.dot(bs, bs))))
    c = np.array(c)
    j = int(np.argmax(c))
    frac = 0.0
    if 0 < j < len(c) - 1:
        frac = 0.5 * (c[j - 1] - c[j + 1]) / (c[j - 1] - 2 * c[j] + c[j + 1])
    return (j - K + frac) / fs


def worst_gap_s(fps_a, fps_b, seconds=1.0):
    """Largest time distance from any frame of A to the nearest frame of B."""
    ta = np.arange(int(seconds * fps_a)) / fps_a
    tb = np.arange(int(seconds * fps_b) + 2) / fps_b
    return float(max(np.min(np.abs(tb - t)) for t in ta))

## See it
Four pictures: the warp, the three aspect fits, resampling blur, and the three interpolants
on a step-like motion.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)

    # 1. homography warp
    img = speckle_image(160, 200, seed=7)
    quad = np.array([[30, 20], [170, 35], [185, 130], [20, 140]], float)       # corners seen by camera B
    grid = np.array([[0, 0], [199, 0], [199, 159], [0, 159]], float)
    H = homography_from_points(quad, grid)
    flat = warp_image(img, H, (160, 200))
    print("homography from 4 point pairs:\n", np.round(H, 4))
    fig, ax = plt.subplots(1, 2, figsize=(8, 3.6))
    ax[0].imshow(img, cmap="gray", vmin=-3, vmax=3)
    ax[0].plot(*np.append(quad, quad[:1], axis=0).T, "r-")
    ax[0].set_title("camera view: sample is a skewed quad")
    ax[1].imshow(flat, cmap="gray", vmin=-3, vmax=3)
    ax[1].set_title("warped onto a 200 x 160 grid")
    for a in ax:
        a.set_xlabel("x (pixels)")
        a.set_ylabel("y (pixels)")
    fig.tight_layout()
    fig.savefig(OUT / "homography_warp.png", dpi=110)

    # 2. aspect fits
    costs = aspect_costs(1280, 720, 640, 480)
    print("1280 x 720 into 640 x 480:", {k: round(v, 4) for k, v in costs.items()})
    fig, ax = plt.subplots(1, 3, figsize=(9, 2.9))
    for a, name in zip(ax, ("letterbox", "crop", "stretch")):
        a.add_patch(plt.Rectangle((0, 0), 640, 480, fc="#dddddd", ec="k"))
        if name == "letterbox":
            a.add_patch(plt.Rectangle((0, 60), 640, 360, fc="#4c78a8"))
            a.set_title(f"letterbox: {costs['letterbox_empty']:.0%} of grid empty")
        elif name == "crop":
            a.add_patch(plt.Rectangle((0, 0), 640, 480, fc="#4c78a8"))
            a.set_title(f"crop: {costs['crop_lost']:.0%} of source lost")
        else:
            a.add_patch(plt.Rectangle((0, 0), 640, 480, fc="#4c78a8"))
            a.set_title(f"stretch: y scaled {costs['stretch_ratio']:.2f}x more than x")
        a.set_xlim(-20, 660)
        a.set_ylim(500, -20)
        a.set_aspect("equal")
        a.set_xlabel("grid x (pixels)")
        a.set_ylabel("grid y (pixels)")
    fig.tight_layout()
    fig.savefig(OUT / "aspect_fits.png", dpi=110)

    # 3. resampling blur
    scales = [0.5, 0.6, 0.75, 0.9, 1.0, 1.25, 1.5, 2.0]
    errs = []
    for s in scales:
        e, s_eff = resample_roundtrip_error(s)
        errs.append(e)
        print(f"scale {s:4.2f}: round-trip error {e:.3f} of the pattern std")
    for f in (0.0, 0.25, 0.5):
        print(f"shift {f:.2f} px: std ratio {shift_std_ratio(f):.4f} (theory {np.hypot(1 - f, f):.4f})")
    fig, ax = plt.subplots(figsize=(5.5, 3.6))
    ax.plot(scales, errs, "o-")
    ax.set_xlabel("scale factor (output size / input size)")
    ax.set_ylabel("round-trip RMS error / pattern std")
    ax.set_title("Bilinear resampling loses detail")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "resample_error.png", dpi=110)

    # 4. interpolants on a step-like motion
    tA, yA = camera_samples(step_motion, 24)
    tq = np.linspace(STEP_WINDOW[0], STEP_WINDOW[1], 600)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(tq, step_motion(tq), "k", lw=2, label="truth")
    for name, col in (("linear", "tab:green"), ("spline", "tab:red"), ("makima", "tab:blue")):
        ax.plot(tq, interpolate(name, tA, yA, tq), color=col, label=name)
    sel = (tA >= 0.3) & (tA <= 0.8)
    ax.plot(tA[sel], yA[sel], "ko", ms=5, label="camera B samples (24 fps)")
    ax.set_xlabel("time (s)")
    ax.set_ylabel("displacement (px)")
    ax.set_title("Step-like motion: spline overshoots, Makima does not")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "step_interpolation.png", dpi=110)

    print("RMS error (px) on the 120 fps timeline:")
    for sig, nm, win in ((smooth_motion, "smooth", (0.0, 2.0)), (step_motion, "step-like", STEP_WINDOW)):
        for fps in (30, 24):
            row = {m: rms_vs_truth(sig, m, fps, win) for m in METHODS}
            print(f"  {nm:9s} {fps} fps:", {k: round(v, 4) for k, v in row.items()})
    for fps in (30, 24):
        print(f"  overshoot at {fps} fps (px):", {m: round(max_overshoot(m, step_motion, fps), 3) for m in METHODS})
    print("  lowest value of the corner curve:", {m: round(corner_dip(m), 4) for m in METHODS})
    tau = 0.0137
    ta, ya = camera_samples(wander_motion, 30)
    tb = np.arange(49) / 24
    yb = wander_motion(tb + tau)
    print(f"true offset {tau * 1000:.1f} ms, estimated {estimate_offset(ta, ya, tb, yb) * 1000:.2f} ms")
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the checkpoint.

## Coding tasks
1. **Easy.** Use `homography_from_points` to map the corners of a 100 x 100 square to the corners of a 100 x 100 square rotated by 10 degrees about its centre and print H.
2. **Stretch.** Add a function that returns the highest value of the `makima` curve through y = 0, 0, 0, 1, 1, 1 and check that it never goes above 1.
3. **Explain.** Explain why the Akima weights give 0/0 when two neighbouring slopes are equal on both sides, and how the extra term in Makima avoids this.

## Assessment
- **A. Conceptual.** Why do we build a warped picture by looking up, for each output pixel, where it came from, instead of moving each source pixel forward?
- **B. Numerical.** Fit a 1280 x 720 camera picture into a 640 x 480 grid. How many rows of the grid stay empty with letterbox, and what fraction of the source is lost with crop?
- **C. Graph and simulation interpretation.** In the interpolation figure, which curve overshoots the step, by how much, and why does RMS error alone not show this?
- **D. Design and optimisation challenge.** Design: camera A is 4:3 and camera B is 16:9; the grid is 4:3 and you cannot afford to lose any of the sample's field of view, but the shape of speckle dots must stay round. Which fit do you choose? And which B frame rate (30 or 60) would you choose if you could?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []

    # ---- homography
    H0 = np.array([[1.0, 0.2, 10.0], [0.0, 1.0, 5.0], [0.001, 0.0, 1.0]])
    q = L.apply_h(H0, [[100.0, 50.0]])[0]
    checks.append((f"worked example: H sends (100, 50) to (109.09, 50.00) (got {q[0]:.2f}, {q[1]:.2f})",
                   abs(q[0] - 120.0 / 1.1) < 1e-9 and abs(q[1] - 50.0) < 1e-9))
    back = L.apply_h(np.linalg.inv(H0), q[None, :])[0]
    checks.append(("the inverse of H sends it back to (100, 50)", np.allclose(back, [100.0, 50.0], atol=1e-9)))
    src = np.array([[0.0, 0.0], [200.0, 0.0], [200.0, 150.0], [0.0, 150.0]])
    dst = L.apply_h(H0, src)
    H1 = L.homography_from_points(src, dst)
    checks.append(("solving the 8 x 8 system from 4 pairs recovers a known H", np.allclose(H1, H0, atol=1e-8)))
    checks.append(("four identical point pairs give the identity", np.allclose(L.homography_from_points(src, src), np.eye(3), atol=1e-9)))
    try:
        L.homography_from_points([[0, 0], [1, 1], [2, 2], [5, 0]], src)
        raised = False
    except ValueError:
        raised = True
    checks.append(("edge: three points on a line are reported as degenerate, not solved", raised))

    # ---- warping and sampling
    img = np.arange(30.0).reshape(5, 6)
    checks.append(("bilinear at a pixel centre returns that pixel", abs(L.bilinear(img, 2.0, 3.0) - img[3, 2]) < 1e-12))
    checks.append(("bilinear halfway between two pixels is their average", abs(L.bilinear(img, 2.5, 3.0) - 0.5 * (img[3, 2] + img[3, 3])) < 1e-12))
    checks.append(("edge: the last pixel column is inside, one step further is NaN",
                   abs(L.bilinear(img, 5.0, 4.0) - img[4, 5]) < 1e-12 and bool(np.isnan(L.bilinear(img, 5.01, 4.0)))))
    sp = L.speckle_image(40, 50, seed=3)
    shifted = L.warp_image(sp, np.array([[1.0, 0, 2.0], [0, 1.0, 0], [0, 0, 1.0]]), sp.shape)
    checks.append(("warping by a 2 pixel translation moves the image exactly", np.allclose(shifted[:, 2:], sp[:, :-2], atol=1e-9) and bool(np.isnan(shifted[0, 0]))))
    quad = np.array([[10.0, 8.0], [60.0, 14.0], [64.0, 50.0], [6.0, 46.0]])
    grid = np.array([[0.0, 0.0], [49.0, 0.0], [49.0, 39.0], [0.0, 39.0]])
    Hq = L.homography_from_points(quad, grid)
    checks.append(("the four quad corners land on the four grid corners", np.allclose(L.apply_h(Hq, quad), grid, atol=1e-8)))
    checks.append(("the speckle pattern has standard deviation about 1 (0.9 to 1.1)", 0.9 < float(L.speckle_image(120, 120).std()) < 1.1))

    # ---- aspect ratio
    c = L.aspect_costs(1280, 720, 640, 480)
    checks.append((f"worked example: 16:9 into 4:3 costs 25 percent (letterbox, crop) or a 1.333 stretch (got {c['letterbox_empty']:.3f}, {c['crop_lost']:.3f}, {c['stretch_ratio']:.3f})",
                   abs(c["letterbox_empty"] - 0.25) < 1e-12 and abs(c["crop_lost"] - 0.25) < 1e-12 and abs(c["stretch_ratio"] - 4 / 3) < 1e-12))
    checks.append(("hand calculation: letterboxing 1280 x 720 into 640 x 480 leaves bars of 0.25 * 480 / 2 = 60 px top and bottom", abs(c["letterbox_empty"] * 480 / 2 - 60) < 1e-9))
    c2 = L.aspect_costs(640, 480, 640, 480)
    checks.append(("edge: equal aspect ratios cost nothing", c2["letterbox_empty"] == 0 and c2["stretch_ratio"] == 1))
    c3 = L.aspect_costs(480, 640, 640, 480)
    checks.append(("a portrait source into a landscape grid also costs 1 - (3/4)/(4/3) = 43.75 percent", abs(c3["letterbox_empty"] - (1 - (3 / 4) / (4 / 3))) < 1e-12))

    # ---- resampling blur
    r5 = L.shift_std_ratio(0.5)
    checks.append((f"half-pixel bilinear shift keeps 0.7071 of the contrast on white noise (got {r5:.3f})", abs(r5 - np.sqrt(0.5)) < 0.01))
    checks.append(("a whole-pixel shift keeps all of it", abs(L.shift_std_ratio(0.0) - 1.0) < 1e-9))
    e = {s: L.resample_roundtrip_error(s)[0] for s in (0.5, 0.75, 0.9, 1.0, 2.0)}
    checks.append((f"scale 1.0 round trip is perfect and scale 0.5 loses about 0.30 of the pattern (got {e[0.5]:.3f})", e[1.0] < 1e-9 and abs(e[0.5] - 0.305) < 0.02))
    checks.append(("the smaller the image gets, the bigger the round-trip error (0.9 < 0.75 < 0.5)", e[0.9] < e[0.75] < e[0.5]))

    # ---- Makima: exactness, continuity, locality
    x = np.arange(11.0)
    xq = np.linspace(0, 10, 1001)
    ln = lambda v: 3 * v - 2
    checks.append(("Makima reproduces a straight line exactly (equal spacing)", np.max(np.abs(L.makima(x, ln(x), xq) - ln(xq))) < 1e-10))
    xn = np.array([0, 1.0, 2.5, 3, 4.6, 6, 7, 8.2, 9, 10])
    checks.append(("Makima reproduces a straight line exactly (uneven spacing)", np.max(np.abs(L.makima(xn, 2 * xn + 1, xq) - (2 * xq + 1))) < 1e-10))
    qd = lambda v: v ** 2
    em = float(np.max(np.abs(L.makima(x, qd(x), xq) - qd(xq))))
    es = float(np.max(np.abs(L.spline(x, qd(x), xq) - qd(xq))))
    ea = float(np.max(np.abs(L.akima(x, qd(x), xq) - qd(xq))))
    checks.append((f"spline and Akima reproduce x^2 exactly but Makima does not: its error is 2/27 = 0.074 (got {em:.4f})",
                   es < 1e-9 and ea < 1e-9 and abs(em - 2 / 27) < 1e-3))
    y = np.sin(x)
    jumps1 = [abs(L.makima(x, y, x[i] - 1e-7, 1) - L.makima(x, y, x[i] + 1e-7, 1)) for i in range(1, 10)]
    jumps2 = [abs(L.makima(x, y, x[i] - 1e-7, 2) - L.makima(x, y, x[i] + 1e-7, 2)) for i in range(1, 10)]
    checks.append(("Makima is C1: the slope has no jump at any knot (below 1e-5)", max(jumps1) < 1e-5))
    checks.append(("Makima is not C2: the curvature jumps at some knot (above 0.1)", max(jumps2) > 0.1))
    checks.append(("Makima passes through every sample", np.allclose(L.makima(x, y, x), y, atol=1e-12)))
    y2 = y.copy()
    y2[0] += 1.0
    cm = np.abs(L.makima(x, y, xq) - L.makima(x, y2, xq)) > 1e-12
    cs = np.abs(L.spline(x, y, xq) - L.spline(x, y2, xq)) > 1e-12
    checks.append(("Makima is local: changing the first sample changes nothing beyond x = 3; the spline changes up to x = 10",
                   xq[cm].max() <= 3.0 + 1e-9 and xq[cs].max() > 9.9))
    checks.append(("edge: flat data give zero slopes, not 0/0", np.allclose(L.akima_slopes(np.arange(5.0), np.ones(5)), 0.0)))
    checks.append(("edge: two samples give a straight line", abs(float(L.makima(np.array([0.0, 1.0]), np.array([0.0, 2.0]), 0.5)) - 1.0) < 1e-12))
    checks.append(("edge: outside the sampled range the answer is NaN unless extrapolation is asked for",
                   bool(np.isnan(L.makima(x, y, 10.5))) and bool(np.isfinite(L.makima(x, y, 10.5, extrapolate=True)))))
    dips = {m: L.corner_dip(m) for m in ("linear", "spline", "akima", "makima")}
    checks.append((f"hand calculation: Akima dips to -0.0741 where a flat run meets a ramp, Makima stays at 0 (got {dips['akima']:.4f}, {dips['makima']:.4f})",
                   abs(dips["akima"] + 0.0741) < 1e-3 and dips["makima"] > -1e-12))
    checks.append((f"the spline dips further to -0.103 (got {dips['spline']:.4f})", abs(dips["spline"] + 0.1026) < 1e-3))

    # ---- time base
    tl = L.timeline(120, 2.0)
    tA, tB = np.arange(61) / 30.0, np.arange(49) / 24.0
    checks.append(("120 fps is a multiple of 30 and 24: every real sample lies on a timeline tick",
                   np.allclose(tl[::4], tA) and np.allclose(tl[::5], tB)))
    sm = {m: L.rms_vs_truth(L.smooth_motion, m, 24) for m in ("linear", "spline", "makima")}
    checks.append((f"smooth motion at 24 fps: spline {sm['spline']:.4f} < Makima {sm['makima']:.4f} < linear {sm['linear']:.4f} px",
                   sm["spline"] < sm["makima"] < sm["linear"] and abs(sm["linear"] - 0.0394) < 0.002 and abs(sm["makima"] - 0.0083) < 0.001))
    st = {m: L.rms_vs_truth(L.step_motion, m, 24, L.STEP_WINDOW) for m in ("linear", "spline", "makima")}
    checks.append((f"step-like motion at 24 fps: Makima {st['makima']:.4f} < linear {st['linear']:.4f} < spline {st['spline']:.4f} px",
                   st["makima"] < st["linear"] < st["spline"] and abs(st["makima"] - 0.0554) < 0.003 and abs(st["spline"] - 0.1471) < 0.005))
    o = {m: L.max_overshoot(m, L.step_motion, 24) for m in ("linear", "spline", "makima")}
    checks.append((f"the spline overshoots the 4 px step by 0.38 px; linear and Makima by 0 (got {o['spline']:.3f}, {o['linear']:.3f}, {o['makima']:.3f})",
                   abs(o["spline"] - 0.383) < 0.01 and o["linear"] < 1e-9 and o["makima"] < 1e-9))

    # ---- offset
    ta, ya = L.camera_samples(L.wander_motion, 30)
    for tau in (0.0137, -0.021):
        yb = L.wander_motion(tB + tau)
        est = L.estimate_offset(ta, ya, tB, yb)
        checks.append((f"offset {tau * 1000:+.1f} ms recovered to within 0.1 ms (got {est * 1000:+.2f} ms)", abs(est - tau) < 1e-4))
    checks.append(("worked example: the worst gap between 30 fps and 24 fps frames is 1/60 s = 16.7 ms", abs(L.worst_gap_s(30, 24) - 1 / 60) < 1e-9))
    checks.append(("edge: cameras at 30 and 60 fps never disagree on frame times (gap 0)", L.worst_gap_s(30, 60) < 1e-12))
    ok_fps = [f for f in (24, 25, 30, 48, 50, 60) if L.worst_gap_s(30, f) <= 0.005]
    checks.append((f"design: of 24, 25, 30, 48, 50, 60 fps only 30 and 60 keep the gap under 5 ms (got {ok_fps})", ok_fps == [30, 60]))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
